# Day 7: Memory

Remember the annoying loop from Day 5? To have a conversation, we had to collect all the messages ourselves and pass them back in on every call. It worked, but in a real app with thousands of users, you'd have to store every user's conversation somewhere, look it up, pass it in, save it again...

Today LangGraph does all of that for us, with two small additions:

- a **checkpointer**, which saves the state after every step
- a **thread_id**, which names the conversation

Think of a checkpointer like the save-game feature in a video game. Every time you finish a level, the game saves your progress. Close the game, come back tomorrow, and you continue exactly where you left off. Each save slot is like a thread: your brother can have his own save slot, and the two never mix.

By the end of today you'll be able to:

- give any graph memory with one extra argument
- keep different conversations separate
- look inside a saved conversation
- save memory to a file so it survives a restart
- give memory to an agent
- keep long conversations small by summarizing them

In [ ]:
%pip install -q -U langgraph langchain langchain-openai langgraph-checkpoint-sqlite python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

In [ ]:
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. One argument that adds memory

We'll take the simple chatbot from Day 5. The only new thing is at compile time: `compile(checkpointer=InMemorySaver())`.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage

In [ ]:
SYSTEM = SystemMessage(content="You are a friendly assistant. Keep answers short.")

In [ ]:
def chatbot(state: MessagesState) -> dict:
    reply = llm.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}

In [ ]:
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("chatbot", chatbot)

In [ ]:
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

In [ ]:
memory = InMemorySaver()

In [ ]:
chat_app = builder.compile(checkpointer=memory)    # this is the only new line

Now, every time we call the graph, we tell it **which conversation** this is, using a `thread_id` inside a `config` dictionary. The structure looks a bit nested, but you'll write it so often it becomes muscle memory.

In [ ]:
config = {"configurable": {"thread_id": "chat-1"}}

In [ ]:
first = chat_app.invoke(
    {"messages": [("user", "Hi! I'm Priya and I work as a data scientist.")]},
    config,
)
print(first["messages"][-1].content)

### Predict before you run

In the next call we send **only** the new question, no history at all. Will the bot know Priya's name and job? And how many messages will be stored in total?

<details>
<summary>Click to see the answer</summary>

Yes, it will know. Because we used the same `thread_id`, LangGraph loaded the saved conversation first, and `add_messages` appended our new question to it.

There will be 4 messages: Priya's first message, the bot's reply, the new question, and the new reply.
</details>

In [ ]:
second = chat_app.invoke({"messages": [("user", "What's my name and job?")]}, config)

print(second["messages"][-1].content)
print("Messages saved in chat-1:", len(second["messages"]))

Compare this with Day 5, where we wrote `history + [new_message]` ourselves. That whole loop is now gone.

Here's what happened behind the scenes on the second call:

```
invoke(new question, thread "chat-1")
   -> load the saved state for "chat-1"           (2 messages)
   -> add the new question with add_messages      (3 messages)
   -> run the chatbot node, append its reply      (4 messages)
   -> save the state for "chat-1" again
```

## 2. Different threads, different conversations

Each `thread_id` is a separate conversation, like separate WhatsApp chats. This is how a single app serves thousands of users: you'd typically use something like the user's id or a session id as the thread id.

In [ ]:
other_config = {"configurable": {"thread_id": "chat-2"}}

result = chat_app.invoke({"messages": [("user", "What's my name?")]}, other_config)

print(result["messages"][-1].content)
print("Messages in chat-2:", len(result["messages"]))

Chat 2 has never heard of Priya. Good, that's exactly what should happen.

### Your turn

Start a third thread with your own name and a hobby, then ask about it in a second call on the same thread. Then ask the same question on a *new* thread id and compare.

In [ ]:
my_config = {"configurable": {"thread_id": "my-thread"}}     # try different ids

chat_app.invoke({"messages": [("user", "Hi, I'm Kiran and I love photography.")]}, my_config)

result = chat_app.invoke({"messages": [("user", "What's my hobby?")]}, my_config)
print(result["messages"][-1].content)

## 3. Looking inside a saved conversation

`get_state(config)` gives you a **snapshot** of a thread: the current values, which node runs next, and some bookkeeping. This is incredibly useful for debugging ("what does the bot actually remember right now?").

In [ ]:
snapshot = chat_app.get_state(config)

print("next node to run:", snapshot.next)          # empty means the run finished
print("step number:     ", snapshot.metadata["step"])
print()
for message in snapshot.values["messages"]:
    print(f"{message.type:5} | {message.content[:70]}")

## 4. Every step is saved

The checkpointer doesn't just keep the latest state. It keeps a save after **every step**, like a game that keeps every save, not just the last one. `get_state_history` lists them, newest first.

This history is what makes tomorrow's lesson possible: going back in time to an earlier step and trying something different.

In [ ]:
for snap in chat_app.get_state_history(config):
    message_count = len(snap.values.get("messages", []))
    print(f"step {snap.metadata['step']:>2} | messages: {message_count} | next: {snap.next}")

## 5. Memory that survives a restart

There's a catch with `InMemorySaver`: as the name says, it lives in memory. Restart the notebook and everything is gone. Fine for learning, not fine for an app.

`SqliteSaver` saves to a file on disk instead. The code barely changes.

A rough guide for later:

- `InMemorySaver` for learning and tests
- `SqliteSaver` (package `langgraph-checkpoint-sqlite`) for small apps and demos
- `PostgresSaver` (package `langgraph-checkpoint-postgres`) for production

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
# check_same_thread=False lets LangGraph use the connection from its worker threads
connection = sqlite3.connect("day07_memory.db", check_same_thread=False)
sqlite_memory = SqliteSaver(connection)

In [ ]:
persistent_app = builder.compile(checkpointer=sqlite_memory)

In [ ]:
saved_config = {"configurable": {"thread_id": "priya-forever"}}

persistent_app.invoke({"messages": [("user", "Remember this: my favourite food is pav bhaji.")]}, saved_config)
print("Saved to day07_memory.db")

Now the fun test. Restart the kernel (Kernel menu, then Restart), run the setup cell at the top, then run the cells in this section from here on. The bot will still remember.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
reloaded_memory = SqliteSaver(sqlite3.connect("day07_memory.db", check_same_thread=False))

In [ ]:
# builder was defined earlier in the notebook. After a restart, re-run the
# chatbot and builder cells from section 1 before this one.
reloaded_app = builder.compile(checkpointer=reloaded_memory)

In [ ]:
result = reloaded_app.invoke(
    {"messages": [("user", "What's my favourite food?")]},
    {"configurable": {"thread_id": "priya-forever"}},
)
print(result["messages"][-1].content)

## 6. Memory for agents

Any graph can have memory, including the tool-using agents from yesterday. With `create_agent`, you just pass `checkpointer=`.

Here's an expense tracker. Notice how the agent can add up the total in the third message, because it remembers the first two.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

In [ ]:
@tool
def add_expense(item: str, amount: float) -> str:
    """Record an expense. The amount is in rupees."""
    return f"Recorded Rs. {amount} for {item}"

In [ ]:
expense_agent = create_agent(
    model=llm,
    tools=[add_expense],
    system_prompt="You track expenses. Use the tool to record each one. You can add up totals from the conversation.",
    checkpointer=InMemorySaver(),
)

In [ ]:
expense_config = {"configurable": {"thread_id": "expenses-october"}}

for text in ["I spent 250 on lunch", "And 120 on an auto ride", "What's my total so far?"]:
    result = expense_agent.invoke({"messages": [("user", text)]}, expense_config)
    print("You:", text)
    print("Bot:", result["messages"][-1].content)
    print()

## 7. Common mistakes

**Mistake 1: Adding a checkpointer but forgetting the thread_id.**
Once a graph has a checkpointer, it needs to know which conversation to save to. Leave out the config and you get an error.

In [ ]:
try:
    chat_app.invoke({"messages": [("user", "hello")]})        # no config passed
except ValueError as error:
    print("Error:", str(error)[:150])

**Fix:** always pass `{"configurable": {"thread_id": "..."}}` as the second argument.

**Mistake 2: Creating a new `InMemorySaver()` every time.**
If you write `builder.compile(checkpointer=InMemorySaver())` inside a function that runs on every message, each call gets a brand-new, empty memory. Create the saver once and reuse it.

**Mistake 3: Passing the whole history while also using a checkpointer.**
Old habits from Day 5. With a checkpointer, send **only the new message**. If you rebuild the history yourself as tuples or new message objects, each one gets a fresh id, so `add_messages` treats them as new and adds them again. The conversation slowly fills up with duplicates.

**Mistake 4: Using the same thread_id for different users.**
If every user shares `"thread-1"`, everyone sees everyone else's conversation. Use something unique per user or per session.

## 8. Keeping long conversations small

With memory, a conversation grows forever. Hundreds of messages means slow responses and a bigger bill on every single turn.

A common solution is the way people take meeting notes. You don't remember every sentence from a two-hour meeting; you remember a short summary plus the last few things that were said.

Here's the plan:

1. After the chatbot replies, check how many messages there are.
2. If there are more than 6, a `summarize` node writes a short summary of the conversation so far.
3. It then deletes the old messages with `RemoveMessage`, keeping only the last 2.
4. The chatbot adds the summary to its system prompt, so nothing important is lost.

```
  START --> chatbot --(more than 6 messages?)--yes--> summarize --> END
                               |
                               no
                               v
                              END
```

In [ ]:
from typing import Literal
from langchain_core.messages import RemoveMessage, HumanMessage

In [ ]:
class SummaryState(MessagesState):
    summary: str

In [ ]:
def chat_with_summary(state: SummaryState) -> dict:
    instructions = "You are a friendly assistant. Keep answers short."

    # If we've summarized before, give the model that summary as background
    if state.get("summary"):
        instructions += f"\nSummary of the earlier conversation: {state['summary']}"

    reply = llm.invoke([SystemMessage(content=instructions)] + state["messages"])
    return {"messages": [reply]}

In [ ]:
def summarize(state: SummaryState) -> dict:
    request = (
        f"Current summary: {state.get('summary', '(none yet)')}\n"
        "Update the summary using the new messages above. Maximum 3 sentences."
    )
    new_summary = llm.invoke(state["messages"] + [HumanMessage(content=request)]).content

    # Delete everything except the last two messages
    old_messages = state["messages"][:-2]
    deletions = [RemoveMessage(id=m.id) for m in old_messages]

    return {"summary": new_summary, "messages": deletions}

In [ ]:
def needs_summary(state: SummaryState) -> Literal["summarize", "__end__"]:
    if len(state["messages"]) > 6:
        return "summarize"
    return END

In [ ]:
builder = StateGraph(SummaryState)

In [ ]:
builder.add_node("chatbot", chat_with_summary)
builder.add_node("summarize", summarize)

In [ ]:
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", needs_summary)
builder.add_edge("summarize", END)

In [ ]:
summary_app = builder.compile(checkpointer=InMemorySaver())
show_graph(summary_app)

### Predict before you run

We'll send four messages on the same thread. Each turn adds 2 messages (yours and the bot's). After the fourth turn, how many messages will be stored, and will there be a summary?

<details>
<summary>Click to see the answer</summary>

After turn 3 there are 6 messages, which is not more than 6, so nothing happens. After turn 4 there are 8, so `summarize` runs: it writes a summary and deletes all but the last 2.

So you'll see **2 messages** stored, plus a summary mentioning Arjun, Nagpur, chess and the dog Bruno.
</details>

In [ ]:
long_config = {"configurable": {"thread_id": "long-chat"}}

for text in ["I'm Arjun from Nagpur.", "I'm learning LangGraph.", "I like chess.", "My dog is called Bruno."]:
    summary_app.invoke({"messages": [("user", text)]}, long_config)

In [ ]:
saved = summary_app.get_state(long_config).values

print("Messages kept:", len(saved["messages"]))
print("Summary:", saved.get("summary"))

Now the real test: ask about something that was in the deleted messages.

In [ ]:
result = summary_app.invoke(
    {"messages": [("user", "What's my dog's name and which city am I from?")]},
    long_config,
)
print(result["messages"][-1].content)

## 9. Practice

**Exercise 1.** Use one compiled chatbot with two threads, `"user-a"` and `"user-b"`. Tell each a different favourite colour, then ask both. Make sure the answers don't mix.

**Exercise 2.** Write a function `count_user_messages(app, thread_id)` that returns how many **human** messages are saved in a thread. A thread that doesn't exist should return 0.

In [ ]:
# Solution 1
builder = StateGraph(MessagesState)

In [ ]:
builder.add_node("chatbot", chatbot)

In [ ]:
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

In [ ]:
colour_app = builder.compile(checkpointer=InMemorySaver())

In [ ]:
user_a = {"configurable": {"thread_id": "user-a"}}
user_b = {"configurable": {"thread_id": "user-b"}}

In [ ]:
colour_app.invoke({"messages": [("user", "My favourite colour is blue.")]}, user_a)
colour_app.invoke({"messages": [("user", "My favourite colour is green.")]}, user_b)

In [ ]:
answer_a = colour_app.invoke({"messages": [("user", "What's my favourite colour?")]}, user_a)
answer_b = colour_app.invoke({"messages": [("user", "What's my favourite colour?")]}, user_b)

print("A:", answer_a["messages"][-1].content)
print("B:", answer_b["messages"][-1].content)

In [ ]:
# Solution 2
def count_user_messages(app, thread_id: str) -> int:
    values = app.get_state({"configurable": {"thread_id": thread_id}}).values
    messages = values.get("messages", [])
    return sum(1 for m in messages if m.type == "human")

In [ ]:
print("user-a sent:", count_user_messages(colour_app, "user-a"))
print("nobody sent:", count_user_messages(colour_app, "nobody"))

assert count_user_messages(colour_app, "user-a") == 2
assert count_user_messages(colour_app, "nobody") == 0

## Wrapping up

A **checkpointer** saves the state after every step, and a **thread_id** says which conversation it belongs to. Together they give any graph memory. You send only the new message; LangGraph loads the rest.

`get_state` shows you what a thread remembers right now, and `get_state_history` shows every saved step. `InMemorySaver` is for learning, `SqliteSaver` and `PostgresSaver` are for real apps. And for long chats, summarize and delete old messages so the cost stays small.

Tomorrow we use those saved steps for something really practical: **pausing** a graph to ask a human for approval, then carrying on. And we'll go back in time to an earlier step and change what happened.